# Day 10 - SVM & KNN

This notebook contains all tasks from Day 10 covering support vector machines, k-nearest neighbors, and dimensionality reduction.

## Task 8: SVM & KNN Comparison - Iris Dataset

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()

X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
}

grid = GridSearchCV(
    SVC(kernel='rbf'),
    param_grid,
    cv=5,
    n_jobs=-1
)

grid.fit(X_tr, y_train)

best_svm = grid.best_estimator_

print("Best SVM Parameters:", grid.best_params_)
print("SVM Test Accuracy:", best_svm.score(X_te, y_test))

k_values = range(1, 21)

train_acc = []
test_acc = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_train)

    train_acc.append(knn.score(X_tr, y_train))
    test_acc.append(knn.score(X_te, y_test))

best_k = list(k_values)[np.argmax(test_acc)]

print("Best k:", best_k)
print("Best KNN Accuracy:", max(test_acc))

plt.figure(figsize=(8, 5))
plt.plot(k_values, train_acc, marker='o', label='Train Accuracy')
plt.plot(k_values, test_acc, marker='o', label='Test Accuracy')
plt.axvline(best_k, linestyle='--', label=f'Best k={best_k}')
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy vs k")
plt.legend()
plt.grid(True)
plt.show()

models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("SVM RBF", best_svm),
    ("KNN", KNeighborsClassifier(n_neighbors=best_k))
]

X_scaled = scaler.fit_transform(X)

for name, model in models:
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    print(f"{name}: Mean={scores.mean():.4f}, Std={scores.std():.4f}")

knn_euclidean = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='euclidean'
)

knn_manhattan = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='manhattan'
)

knn_euclidean.fit(X_tr, y_train)
knn_manhattan.fit(X_tr, y_train)

euclidean_acc = knn_euclidean.score(X_te, y_test)
manhattan_acc = knn_manhattan.score(X_te, y_test)

print("Euclidean Accuracy:", euclidean_acc)
print("Manhattan Accuracy:", manhattan_acc)

if euclidean_acc > manhattan_acc:
    print("Euclidean performs better")
elif manhattan_acc > euclidean_acc:
    print("Manhattan performs better")
else:
    print("Both perform equally")

## Task 9: Curse of Dimensionality - KNN with Noise

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import numpy as np

X, y = make_classification(
    n_samples=500,
    n_features=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

original_acc = accuracy_score(y_test, y_pred)

print("Accuracy with 2 features:", original_acc)

np.random.seed(42)

noise = np.random.randn(500, 50)

X_noisy = np.hstack([X, noise])

Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    X_noisy, y, test_size=0.2, random_state=42
)

scaler_noisy = StandardScaler()

Xn_train_scaled = scaler_noisy.fit_transform(Xn_train)
Xn_test_scaled = scaler_noisy.transform(Xn_test)

knn_noisy = KNeighborsClassifier(n_neighbors=5)

knn_noisy.fit(Xn_train_scaled, yn_train)

yn_pred = knn_noisy.predict(Xn_test_scaled)

noisy_acc = accuracy_score(yn_test, yn_pred)

print("Accuracy with 52 features:", noisy_acc)

drop = original_acc - noisy_acc

print("Accuracy Drop:", drop)

pca = PCA(n_components=2)

Xn_train_pca = pca.fit_transform(Xn_train_scaled)
Xn_test_pca = pca.transform(Xn_test_scaled)

knn_pca = KNeighborsClassifier(n_neighbors=5)

knn_pca.fit(Xn_train_pca, yn_train)

pca_pred = knn_pca.predict(Xn_test_pca)

pca_acc = accuracy_score(yn_test, pca_pred)

print("Accuracy after PCA:", pca_acc)

if pca_acc > noisy_acc:
    print("Accuracy recovered after PCA")

print("This demonstrates that KNN struggles in high dimensions because distance becomes less meaningful when many irrelevant features are added.")